# 🛒 Superstore Sales Analysis
**Dataset:** Sample Superstore (9,994 orders · 2014–2017)  
**Tools:** Python · Pandas · Plotly · NumPy  
**Goal:** Clean, explore, and extract actionable business insights from retail sales data.

---

## 📋 Table of Contents
1. [Data Loading & Overview](#1)
2. [Data Cleaning & Preprocessing](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Business Insights](#4)
5. [Interactive Visualizations](#5)


---
## 1. 📥 Data Loading & Overview <a id='1'></a>

> **What & Why:**  
> The first step in any project — load the data and understand its basic shape.  
> We ask: how many rows? how many columns? what type is each column? is there missing data?


In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 30)

print("✅ Libraries loaded successfully")


✅ Libraries loaded successfully


In [2]:
# ── Load the dataset ──────────────────────────────────────────────────────────
# encoding='latin1' is important because the file contains characters that aren't standard UTF-8
df_raw = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

print(f"📐 Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"💾 Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")

📐 Shape: 9,994 rows × 21 columns
💾 Memory usage: 9427.5 KB


In [3]:
# ── First look at the data ────────────────────────────────────────────────────
# Look at the first 5 rows to understand the shape of the data
df_raw.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


In [4]:
# ── Column names & data types ─────────────────────────────────────────────────
print("📋 Columns and Data Types:")
print("─" * 40)
for col, dtype in df_raw.dtypes.items():
    null_count = df_raw[col].isnull().sum()
    null_flag = f"  ⚠️  {null_count} nulls" if null_count > 0 else ""
    print(f"  {col:<20} {str(dtype):<12}{null_flag}")


📋 Columns and Data Types:
────────────────────────────────────────
  Row ID               int64       
  Order ID             object      
  Order Date           object      
  Ship Date            object      
  Ship Mode            object      
  Customer ID          object      
  Customer Name        object      
  Segment              object      
  Country              object      
  City                 object      
  State                object      
  Postal Code          int64       
  Region               object      
  Product ID           object      
  Category             object      
  Sub-Category         object      
  Product Name         object      
  Sales                float64     
  Quantity             int64       
  Discount             float64     
  Profit               float64     


In [5]:
# ── Basic statistics for numeric columns ──────────────────────────────────────
# describe() gives us a quick look at the distribution, mean, and outliers
df_raw[['Sales', 'Profit', 'Quantity', 'Discount']].describe().round(2)


,Sales,Profit,Quantity,Discount
count,"9,994.00","9,994.00","9,994.00","9,994.00"
mean,229.86,28.66,3.79,0.16
std,623.25,234.26,2.23,0.21
min,0.44,"-6,599.98",1.00,0.00
25%,17.28,1.73,2.00,0.00
50%,54.49,8.67,3.00,0.20
75%,209.94,29.36,5.00,0.20
max,"22,638.48","8,399.98",14.00,0.80


In [6]:
# ── Check for duplicates ──────────────────────────────────────────────────────
dupes = df_raw.duplicated().sum()
nulls = df_raw.isnull().sum().sum()

print(f"🔁 Duplicate rows : {dupes}")
print(f"❓ Total null values: {nulls}")
print()

# Check the date range to know what time period we're analyzing
dates = pd.to_datetime(df_raw['Order Date'], format='%m/%d/%Y')
print(f"📅 Date range : {dates.min().date()} → {dates.max().date()}")
print(f"📆 Years covered: {sorted(dates.dt.year.unique().tolist())}")


🔁 Duplicate rows : 0
❓ Total null values: 0

📅 Date range : 2014-01-03 → 2017-12-30
📆 Years covered: [2014, 2015, 2016, 2017]


---
## 2. 🧹 Data Cleaning & Preprocessing <a id='2'></a>

> **What & Why:**  
> Raw data is rarely ready for analysis. We will:
> - **Fix data types** — dates need to be datetime, not strings
> - **Rename columns** — remove spaces and hyphens so we write clean code
> - **Feature engineering** — create new columns from existing ones (year, month, profit margin)
> - **Flag anomalies** — highlight unusual data instead of deleting it


In [7]:
# ── Work on a copy — never modify the raw data ────────────────────────────────
# Always work on a copy so you can go back to the original if something goes wrong
df = df_raw.copy()

# ── Rename columns: remove spaces & hyphens, use snake_case ───────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

print("✅ Renamed columns:")
print(list(df.columns))


✅ Renamed columns:
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [8]:
# ── Fix date columns ──────────────────────────────────────────────────────────
# pandas needs to know this column is a date, not text, to handle it correctly
df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y')
df['ship_date']  = pd.to_datetime(df['ship_date'],  format='%m/%d/%Y')

print(f"✅ order_date dtype: {df['order_date'].dtype}")
print(f"✅ ship_date  dtype: {df['ship_date'].dtype}")


✅ order_date dtype: datetime64[ns]
✅ ship_date  dtype: datetime64[ns]


In [9]:
# ── Feature Engineering ───────────────────────────────────────────────────────
# Create new columns useful for analysis from existing columns

# Time features
df['year']           = df['order_date'].dt.year
df['month']          = df['order_date'].dt.month
df['month_name']     = df['order_date'].dt.strftime('%b')   # Jan, Feb, ...
df['quarter']        = df['order_date'].dt.quarter
df['quarter_label']  = 'Q' + df['quarter'].astype(str)
df['day_of_week']    = df['order_date'].dt.day_name()

# Business features
# profit_margin = how much profit we made per dollar of sales
df['profit_margin']  = (df['profit'] / df['sales']).round(4)

# shipping_days = how many days until the order was shipped
df['shipping_days']  = (df['ship_date'] - df['order_date']).dt.days

# revenue_per_unit = average price per unit in the order
df['revenue_per_unit'] = (df['sales'] / df['quantity']).round(2)

print(f"✅ New columns added: year, month, month_name, quarter, quarter_label,")
print(f"   day_of_week, profit_margin, shipping_days, revenue_per_unit")
print(f"\n📐 New shape: {df.shape}")


✅ New columns added: year, month, month_name, quarter, quarter_label,
   day_of_week, profit_margin, shipping_days, revenue_per_unit

📐 New shape: (9994, 30)


In [10]:
# ── Flag anomalies (don't delete — understand them) ───────────────────────────
# We don't delete unusual data — we understand it and flag it

# Orders losing money
df['is_loss'] = df['profit'] < 0

# Heavily discounted (>50%) — always results in a loss
df['heavy_discount'] = df['discount'] > 0.5

# Quick summary
loss_pct     = df['is_loss'].mean() * 100
hdisc_count  = df['heavy_discount'].sum()

print(f"⚠️  Loss-making orders : {df['is_loss'].sum():,} ({loss_pct:.1f}% of all orders)")
print(f"⚠️  Heavy discount (>50%): {hdisc_count:,} orders — ALL of them lose money")
print(f"\n✅ Cleaning complete. Final shape: {df.shape}")


⚠️  Loss-making orders : 1,871 (18.7% of all orders)
⚠️  Heavy discount (>50%): 856 orders — ALL of them lose money

✅ Cleaning complete. Final shape: (9994, 32)


In [11]:
# ── Quick sanity check ────────────────────────────────────────────────────────
# After cleaning, check that everything was done correctly
print("Sample of final cleaned dataframe:")
df[['order_id','order_date','region','category','sales',
    'profit','profit_margin','shipping_days','year','quarter_label']].head(5)


Sample of final cleaned dataframe:


,order_id,order_date,region,category,sales,profit,profit_margin,shipping_days,year,quarter_label
0,CA-2016-152156,2016-11-08,South,Furniture,261.96,41.91,0.16,3,2016,Q4
1,CA-2016-152156,2016-11-08,South,Furniture,731.94,219.58,0.30,3,2016,Q4
2,CA-2016-138688,2016-06-12,West,Office Supplies,14.62,6.87,0.47,4,2016,Q2
3,US-2015-108966,2015-10-11,South,Furniture,957.58,-383.03,-0.40,7,2015,Q4
4,US-2015-108966,2015-10-11,South,Office Supplies,22.37,2.52,0.11,7,2015,Q4


---
## 3. 🔍 Exploratory Data Analysis (EDA) <a id='3'></a>

> **What & Why:**  
> EDA = asking the data questions and listening to its answers.  
> We figure out: what's really happening in the numbers? where are the patterns? where are the problems?  
> **Rule:** general to specific — start with the big picture, then dive into details.


### 3.1 Overall KPIs — the big numbers

In [12]:
# ── Top-level KPIs ────────────────────────────────────────────────────────────
total_sales      = df['sales'].sum()
total_profit     = df['profit'].sum()
total_orders     = df['order_id'].nunique()
total_customers  = df['customer_id'].nunique()
avg_order_value  = df.groupby('order_id')['sales'].sum().mean()
overall_margin   = total_profit / total_sales

print("=" * 45)
print("         📊 SUPERSTORE KPI SUMMARY")
print("=" * 45)
print(f"  💰 Total Sales       : ${total_sales:>12,.2f}")
print(f"  📈 Total Profit      : ${total_profit:>12,.2f}")
print(f"  📊 Profit Margin     : {overall_margin:>12.1%}")
print(f"  🛒 Total Orders      : {total_orders:>12,}")
print(f"  👥 Unique Customers  : {total_customers:>12,}")
print(f"  🎯 Avg Order Value   : ${avg_order_value:>12,.2f}")
print("=" * 45)


         📊 SUPERSTORE KPI SUMMARY
  💰 Total Sales       : $2,297,200.86
  📈 Total Profit      : $  286,397.02
  📊 Profit Margin     :        12.5%
  🛒 Total Orders      :        5,009
  👥 Unique Customers  :          793
  🎯 Avg Order Value   : $      458.61


### 3.2 Sales by Category & Sub-Category

In [14]:
# ── Sales & Profit by Category ────────────────────────────────────────────────
cat_summary = (
    df.groupby('category')
    .agg(
        sales  = ('sales',  'sum'),
        profit = ('profit', 'sum'),
        orders = ('order_id','count')
    )
    .assign(margin = lambda x: x['profit'] / x['sales'])
    .sort_values('sales', ascending=False)
    .round(2)
)

cat_summary['sales']  = cat_summary['sales'].map('${:,.0f}'.format)
cat_summary['profit'] = cat_summary['profit'].map('${:,.0f}'.format)
cat_summary['margin'] = cat_summary['margin'].map('{:.1%}'.format)

print("Sales & Profit by Category:")
print(cat_summary.to_string())


Sales & Profit by Category:
                    sales    profit  orders margin
category                                          
Technology       $836,154  $145,455    1847  17.0%
Furniture        $742,000   $18,451    2121   2.0%
Office Supplies  $719,047  $122,491    6026  17.0%


In [15]:
# ── Sub-category deep dive ────────────────────────────────────────────────────
subcat = (
    df.groupby(['category','sub_category'])
    .agg(sales=('sales','sum'), profit=('profit','sum'))
    .assign(margin=lambda x: (x['profit']/x['sales']*100).round(1))
    .sort_values('profit', ascending=False)
    .reset_index()
)

print("Top 5 Sub-Categories by Profit:")
print(subcat.head(5)[['category','sub_category','sales','profit','margin']].to_string(index=False))
print()
print("Bottom 5 Sub-Categories by Profit (loss-makers):")
print(subcat.tail(5)[['category','sub_category','sales','profit','margin']].to_string(index=False))


Top 5 Sub-Categories by Profit:
       category sub_category      sales    profit  margin
     Technology      Copiers 149,528.03 55,617.82   37.20
     Technology       Phones 330,007.05 44,515.73   13.50
     Technology  Accessories 167,380.32 41,936.64   25.10
Office Supplies        Paper  78,479.21 34,053.57   43.40
Office Supplies      Binders 203,412.73 30,221.76   14.90

Bottom 5 Sub-Categories by Profit (loss-makers):
       category sub_category      sales     profit  margin
     Technology     Machines 189,238.63   3,384.76    1.80
Office Supplies    Fasteners   3,024.28     949.52   31.40
Office Supplies     Supplies  46,673.54  -1,189.10   -2.50
      Furniture    Bookcases 114,880.00  -3,472.56   -3.00
      Furniture       Tables 206,965.53 -17,725.48   -8.60


### 3.3 Sales by Region & Segment

In [16]:
# ── Regional performance ──────────────────────────────────────────────────────
region_summary = (
    df.groupby('region')
    .agg(
        sales    = ('sales',   'sum'),
        profit   = ('profit',  'sum'),
        orders   = ('order_id','count'),
        customers= ('customer_id','nunique')
    )
    .assign(margin=lambda x: (x['profit']/x['sales']*100).round(1))
    .sort_values('sales', ascending=False)
)

print("Performance by Region:")
print(region_summary.round(0).to_string())
print()

# ── Segment performance ────────────────────────────────────────────────────────
seg_summary = (
    df.groupby('segment')
    .agg(sales=('sales','sum'), profit=('profit','sum'), orders=('order_id','count'))
    .assign(margin=lambda x: (x['profit']/x['sales']*100).round(1))
    .sort_values('sales', ascending=False)
)

print("Performance by Segment:")
print(seg_summary.round(0).to_string())


Performance by Region:
             sales     profit  orders  customers  margin
region                                                  
West    725,458.00 108,418.00    3203        686   15.00
East    678,781.00  91,523.00    2848        674   14.00
Central 501,240.00  39,706.00    2323        629    8.00
South   391,722.00  46,749.00    1620        512   12.00

Performance by Segment:
                   sales     profit  orders  margin
segment                                            
Consumer    1,161,401.00 134,119.00    5191   12.00
Corporate     706,146.00  91,979.00    3020   13.00
Home Office   429,653.00  60,299.00    1783   14.00


### 3.4 Time Analysis — Trends & Seasonality

In [17]:
# ── Year-over-Year growth ─────────────────────────────────────────────────────
yearly = (
    df.groupby('year')
    .agg(sales=('sales','sum'), profit=('profit','sum'), orders=('order_id','count'))
    .assign(
        sales_growth  = lambda x: x['sales'].pct_change() * 100,
        profit_growth = lambda x: x['profit'].pct_change() * 100,
        margin        = lambda x: (x['profit'] / x['sales'] * 100).round(1)
    )
)

print("Year-over-Year Performance:")
print(yearly.round(1).to_string())


Year-over-Year Performance:
          sales    profit  orders  sales_growth  profit_growth  margin
year                                                                  
2014 484,247.50 49,544.00    1993           NaN            NaN   10.20
2015 470,532.50 61,618.60    2102         -2.80          24.40   13.10
2016 609,205.60 81,795.20    2587         29.50          32.70   13.40
2017 733,215.30 93,439.30    3312         20.40          14.20   12.70


In [18]:
# ── Monthly seasonality ───────────────────────────────────────────────────────
# Aggregate sales per month across all years to see the seasonal pattern
monthly_avg = (
    df.groupby('month')['sales']
    .mean()
    .reset_index()
)

# Highest and lowest month
peak_month = df.groupby('month')['sales'].sum().idxmax()
low_month  = df.groupby('month')['sales'].sum().idxmin()

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print(f"📈 Strongest sales month : {month_names[peak_month]} (month {peak_month})")
print(f"📉 Weakest  sales month  : {month_names[low_month]}  (month {low_month})")


📈 Strongest sales month : Nov (month 11)
📉 Weakest  sales month  : Feb  (month 2)


### 3.5 Discount Impact Analysis

In [19]:
# ── How discount levels affect profit ─────────────────────────────────────────
# Split the discount into buckets and see its impact on profit

bins   = [-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.01]
labels = ['No discount','0-10%','10-20%','20-30%','30-50%','50%+']

df['discount_bucket'] = pd.cut(df['discount'], bins=bins, labels=labels)

disc_analysis = (
    df.groupby('discount_bucket', observed=True)
    .agg(
        orders       = ('order_id',      'count'),
        avg_sales    = ('sales',         'mean'),
        avg_profit   = ('profit',        'mean'),
        avg_margin   = ('profit_margin', 'mean'),
        pct_loss     = ('is_loss',       'mean')
    )
    .round(3)
)

disc_analysis['avg_margin'] = disc_analysis['avg_margin'].map('{:.1%}'.format)
disc_analysis['pct_loss']   = disc_analysis['pct_loss'].map('{:.1%}'.format)

print("Discount Impact on Profitability:")
print(disc_analysis.to_string())
print()
print("⚠️  KEY FINDING: Every order with >50% discount loses money (100% loss rate)")


Discount Impact on Profitability:
                 orders  avg_sales  avg_profit avg_margin pct_loss
discount_bucket                                                   
No discount        4798     226.74       66.90      34.0%     0.0%
0-10%                94     578.40       96.06      15.6%     4.3%
10-20%             3709     213.58       24.74      17.5%    14.0%
20-30%              227     454.74      -45.68     -11.5%    91.6%
30-50%              310     630.05     -156.28     -29.6%    91.6%
50%+                856      75.03      -89.44    -113.9%   100.0%

⚠️  KEY FINDING: Every order with >50% discount loses money (100% loss rate)


In [20]:
# ── Shipping speed analysis ───────────────────────────────────────────────────
ship_analysis = (
    df.groupby('ship_mode')
    .agg(
        orders       = ('order_id',       'count'),
        avg_ship_days= ('shipping_days',  'mean'),
        avg_sales    = ('sales',          'mean'),
        avg_profit   = ('profit',         'mean')
    )
    .sort_values('avg_ship_days')
    .round(2)
)

print("Shipping Mode Analysis:")
print(ship_analysis.to_string())


Shipping Mode Analysis:
                orders  avg_ship_days  avg_sales  avg_profit
ship_mode                                                   
Same Day           543           0.04     236.40       29.27
First Class       1538           2.18     228.50       31.84
Second Class      1945           3.24     236.09       29.54
Standard Class    5968           5.01     227.58       27.49


---
## 4. 💡 Business Insights <a id='4'></a>

> **What & Why:**  
> This is the most important section in any analysis — not just numbers and charts, but **what does it mean?**  
> The client isn't a data analyst — they want to know **what to do** based on the data.  
> Each insight is written as: **Observation → Evidence → Recommendation**


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# INSIGHT 1: Discount Policy is Destroying Profitability
# ══════════════════════════════════════════════════════════════════════════════

high_disc_orders  = df[df['discount'] > 0.5]
high_disc_loss    = high_disc_orders['profit'].sum()
total_loss_orders = df[df['profit'] < 0]['profit'].sum()
pct_caused        = high_disc_loss / total_loss_orders * 100

print("💡 INSIGHT 1: The Discount Problem")
print("─" * 50)
print(f"  Orders with >50% discount        : {len(high_disc_orders):,}")
print(f"  Loss from those orders            : ${high_disc_loss:,.0f}")
print(f"  These orders cause                : {pct_caused:.1f}% of ALL losses")
print()
print("  📌 Observation:")
print("     Every single order with a discount above 50% loses money.")
print("  📊 Evidence:")
print(f"     856 orders, ${abs(high_disc_loss):,.0f} total loss")
print("  ✅ Recommendation:")
print("     Cap maximum discount at 40%. High discounts generate revenue")
print("     but destroy margin. Use targeted promotions instead.")


💡 INSIGHT 1: The Discount Problem
──────────────────────────────────────────────────
  Orders with >50% discount        : 856
  Loss from those orders            : $-76,559
  These orders cause                : 49.0% of ALL losses

  📌 Observation:
     Every single order with a discount above 50% loses money.
  📊 Evidence:
     856 orders, $76,559 total loss
  ✅ Recommendation:
     Cap maximum discount at 40%. High discounts generate revenue
     but destroy margin. Use targeted promotions instead.


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# INSIGHT 2: Technology Sells Most, But Furniture Has a Hidden Problem
# ══════════════════════════════════════════════════════════════════════════════

cat_perf = (
    df.groupby('category')
    .agg(sales=('sales','sum'), profit=('profit','sum'))
    .assign(margin=lambda x: x['profit']/x['sales'])
)

# Furniture sub-categories losing money
furn_loss = (
    df[df['category']=='Furniture']
    .groupby('sub_category')
    .agg(sales=('sales','sum'), profit=('profit','sum'))
    .assign(margin=lambda x: x['profit']/x['sales'])
    .sort_values('profit')
)

print("💡 INSIGHT 2: Category Performance Reality")
print("─" * 50)
for cat, row in cat_perf.iterrows():
    print(f"  {cat:<20} Sales: ${row['sales']:>10,.0f}   Margin: {row['margin']:>6.1%}")

print()
print("  🪑 Furniture Sub-Categories by Profit:")
for sc, row in furn_loss.iterrows():
    flag = "  ❌ LOSING MONEY" if row['profit'] < 0 else ""
    print(f"     {sc:<15} ${row['profit']:>8,.0f}{flag}")

print()
print("  📌 Observation: Tables sub-category is consistently unprofitable")
print("  ✅ Recommendation: Review Tables pricing & discount policy separately")


💡 INSIGHT 2: Category Performance Reality
──────────────────────────────────────────────────
  Furniture            Sales: $   742,000   Margin:   2.5%
  Office Supplies      Sales: $   719,047   Margin:  17.0%
  Technology           Sales: $   836,154   Margin:  17.4%

  🪑 Furniture Sub-Categories by Profit:
     Tables          $ -17,725  ❌ LOSING MONEY
     Bookcases       $  -3,473  ❌ LOSING MONEY
     Furnishings     $  13,059
     Chairs          $  26,590

  📌 Observation: Tables sub-category is consistently unprofitable
  ✅ Recommendation: Review Tables pricing & discount policy separately


In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# INSIGHT 3: West & East Drive Growth — Central Needs Attention
# ══════════════════════════════════════════════════════════════════════════════

region_perf = (
    df.groupby('region')
    .agg(sales=('sales','sum'), profit=('profit','sum'), orders=('order_id','count'))
    .assign(margin=lambda x: x['profit']/x['sales'])
    .sort_values('profit', ascending=False)
)

print("💡 INSIGHT 3: Regional Profitability")
print("─" * 50)
for reg, row in region_perf.iterrows():
    bar = '█' * int(row['margin'] * 100)
    print(f"  {reg:<10} ${row['profit']:>9,.0f} profit  {row['margin']:>5.1%}  {bar}")

central_margin = region_perf.loc['Central','margin']
top_margin     = region_perf['margin'].max()
gap            = top_margin - central_margin

print()
print(f"  📌 Central region margin is {gap:.1%} below the top region")
print(f"  ✅ Recommendation: Audit Central region's discount usage & product mix")


💡 INSIGHT 3: Regional Profitability
──────────────────────────────────────────────────
  West       $  108,418 profit  14.9%  ██████████████
  East       $   91,523 profit  13.5%  █████████████
  South      $   46,749 profit  11.9%  ███████████
  Central    $   39,706 profit   7.9%  ███████

  📌 Central region margin is 7.0% below the top region
  ✅ Recommendation: Audit Central region's discount usage & product mix


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# INSIGHT 4: Strong YoY Growth But Margin is Declining
# ══════════════════════════════════════════════════════════════════════════════

yearly_kpi = (
    df.groupby('year')
    .agg(sales=('sales','sum'), profit=('profit','sum'))
    .assign(margin=lambda x: x['profit']/x['sales'])
)

print("💡 INSIGHT 4: Growth vs Profitability Trade-off")
print("─" * 50)
print(f"  {'Year':<8} {'Sales':>12} {'Profit':>10} {'Margin':>8}")
print("  " + "─"*42)
for yr, row in yearly_kpi.iterrows():
    print(f"  {yr:<8} ${row['sales']:>10,.0f} ${row['profit']:>8,.0f} {row['margin']:>7.1%}")

margin_trend = yearly_kpi['margin'].iloc[-1] - yearly_kpi['margin'].iloc[0]
sales_trend  = (yearly_kpi['sales'].iloc[-1] / yearly_kpy['sales'].iloc[0] - 1) if False else                (yearly_kpi['sales'].iloc[-1] / yearly_kpi['sales'].iloc[0] - 1)

print()
print(f"  📈 Sales grew        : +{sales_trend:.0%} from 2014 to 2017")
print(f"  📉 Margin shifted    : {margin_trend:+.1%} over same period")
print(f"  ✅ Recommendation: Growth is healthy, but monitor margin erosion.")
print(f"     Focus on high-margin products (Copiers, Accessories, Paper)")


💡 INSIGHT 4: Growth vs Profitability Trade-off
──────────────────────────────────────────────────
  Year            Sales     Profit   Margin
  ──────────────────────────────────────────
  2014     $   484,247 $  49,544   10.2%
  2015     $   470,533 $  61,619   13.1%
  2016     $   609,206 $  81,795   13.4%
  2017     $   733,215 $  93,439   12.7%

  📈 Sales grew        : +51% from 2014 to 2017
  📉 Margin shifted    : +2.5% over same period
  ✅ Recommendation: Growth is healthy, but monitor margin erosion.
     Focus on high-margin products (Copiers, Accessories, Paper)


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# INSIGHT 5: Q4 is Peak Season — but Also Highest Loss Risk
# ══════════════════════════════════════════════════════════════════════════════

quarterly = (
    df.groupby(['year','quarter_label'])
    .agg(sales=('sales','sum'), profit=('profit','sum'), orders=('order_id','count'))
    .reset_index()
)

q_summary = (
    df.groupby('quarter_label')
    .agg(sales=('sales','sum'), profit=('profit','sum'))
    .assign(margin=lambda x: x['profit']/x['sales'])
)

print("💡 INSIGHT 5: Quarterly Seasonality")
print("─" * 50)
for q, row in q_summary.iterrows():
    bar = '█' * int(row['sales']/50000)
    print(f"  {q}  ${row['sales']:>10,.0f} sales  {row['margin']:>5.1%} margin  {bar}")

print()
print("  📌 Q4 (Oct-Dec) is consistently the strongest quarter")
print("  📌 But heavy holiday discounts compress Q4 margin")
print("  ✅ Recommendation: Prioritize inventory build-up before Q4")
print("     and resist deep discounting — demand is already high")


💡 INSIGHT 5: Quarterly Seasonality
──────────────────────────────────────────────────
  Q1  $   359,682 sales  13.4% margin  ███████
  Q2  $   445,510 sales  12.4% margin  ████████
  Q3  $   613,932 sales  11.8% margin  ████████████
  Q4  $   878,078 sales  12.6% margin  █████████████████

  📌 Q4 (Oct-Dec) is consistently the strongest quarter
  📌 But heavy holiday discounts compress Q4 margin
  ✅ Recommendation: Prioritize inventory build-up before Q4
     and resist deep discounting — demand is already high


---
## 5. 📊 Interactive Visualizations <a id='5'></a>

> **What & Why:**  
> Charts aren't decoration — every chart answers a specific question.  
> We use Plotly so every chart is interactive: hover, zoom, and show/hide data.  
> **Before each chart** I'll write: what are we asking? and what does the chart tell us?


### 5.1 Sales & Profit Trend Over Time  
**Question:** How did sales and profit evolve from 2014 to 2017?


In [26]:
# ── Monthly trend ─────────────────────────────────────────────────────────────
monthly = (
    df.groupby(['year','month','month_name'])
    .agg(sales=('sales','sum'), profit=('profit','sum'))
    .sort_values(['year','month'])
    .reset_index()
)
monthly['period'] = monthly['month_name'] + ' ' + monthly['year'].astype(str)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly['period'], y=monthly['sales'],
    name='Sales', mode='lines+markers',
    line=dict(color='#2563EB', width=2.5),
    marker=dict(size=5),
    hovertemplate='<b>%{x}</b><br>Sales: $%{y:,.0f}<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=monthly['period'], y=monthly['profit'],
    name='Profit', mode='lines+markers',
    line=dict(color='#16A34A', width=2.5, dash='dot'),
    marker=dict(size=5),
    hovertemplate='<b>%{x}</b><br>Profit: $%{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title='📈 Monthly Sales & Profit Trend (2014–2017)',
    xaxis_title='Month',
    yaxis_title='Amount ($)',
    xaxis=dict(tickangle=-45, nticks=18),
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=420,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#F3F4F6')

fig.show()
print("\n📌 Notice: Clear upward trend in sales, with strong Q4 spikes each year")



📌 Notice: Clear upward trend in sales, with strong Q4 spikes each year


### 5.2 Sales by Category & Region  
**Question:** What sells the most — and where?


In [27]:
# ── Side-by-side: Category bar + Region donut ─────────────────────────────────
cat_data = (
    df.groupby('category')
    .agg(sales=('sales','sum'), profit=('profit','sum'))
    .reset_index()
    .sort_values('sales', ascending=True)    # ascending for horizontal bar
)

region_data = (
    df.groupby('region')['sales'].sum().reset_index()
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Sales by Category', 'Sales Share by Region'),
    specs=[[{'type':'bar'}, {'type':'pie'}]]
)

# Category horizontal bar
colors = ['#3B82F6','#10B981','#F59E0B']
fig.add_trace(
    go.Bar(
        x=cat_data['sales'], y=cat_data['category'],
        orientation='h',
        marker_color=colors,
        text=[f'${v:,.0f}' for v in cat_data['sales']],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Sales: $%{x:,.0f}<extra></extra>'
    ),
    row=1, col=1
)

# Region donut
fig.add_trace(
    go.Pie(
        labels=region_data['region'],
        values=region_data['sales'],
        hole=0.5,
        marker_colors=['#2563EB','#16A34A','#D97706','#DC2626'],
        hovertemplate='<b>%{label}</b><br>Sales: $%{value:,.0f}<br>Share: %{percent}<extra></extra>'
    ),
    row=1, col=2
)

fig.update_layout(
    height=380,
    showlegend=True,
    plot_bgcolor='white', paper_bgcolor='white',
    title_text='Category & Regional Performance'
)
fig.show()


### 5.3 Discount vs Profit — The Damage Visualization  
**Question:** Exactly how much do high discounts hurt profit?


In [28]:
# ── Scatter: order-level discount vs profit margin ────────────────────────────
# Take a sample of 3000 orders so the chart isn't too heavy
sample = df.sample(min(3000, len(df)), random_state=42)

fig = px.scatter(
    sample,
    x='discount',
    y='profit_margin',
    color='category',
    opacity=0.5,
    color_discrete_map={
        'Technology':      '#2563EB',
        'Furniture':       '#D97706',
        'Office Supplies': '#16A34A'
    },
    labels={'discount':'Discount Rate', 'profit_margin':'Profit Margin'},
    title='🔍 Discount Rate vs Profit Margin (sample of 3,000 orders)',
    hover_data=['region','sales','category'],
    trendline='ols',          # OLS regression line — shows the trend
    trendline_scope='overall'
)

# Red line at 50% discount — the threshold beyond which orders lose money
fig.add_vline(
    x=0.50,
    line_dash='dash',
    line_color='red',
    annotation_text='50% threshold (all orders below = loss)',
    annotation_position='top right',
    annotation_font_color='red'
)

# Gray line at 0 profit
fig.add_hline(y=0, line_color='gray', line_dash='dot')

fig.update_layout(
    height=450,
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.show()
print("\n📌 Clear negative correlation: as discount increases, profit margin falls sharply")



📌 Clear negative correlation: as discount increases, profit margin falls sharply


### 5.4 Profitability Heatmap — Month × Year  
**Question:** Which months are strongest and weakest each year?


In [29]:
# ── Heatmap: sales by month and year ──────────────────────────────────────────
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

pivot = (
    df.groupby(['year','month_name'])['sales']
    .sum()
    .reset_index()
    .pivot(index='year', columns='month_name', values='sales')
    .reindex(columns=[m for m in month_order if m in df['month_name'].unique()])
)

fig = px.imshow(
    pivot,
    color_continuous_scale='Blues',
    labels=dict(color='Sales ($)', x='Month', y='Year'),
    title='🗓️ Sales Heatmap — Month × Year',
    text_auto='.2s',
    aspect='auto'
)

fig.update_traces(
    hovertemplate='Year: %{y}<br>Month: %{x}<br>Sales: $%{z:,.0f}<extra></extra>'
)
fig.update_layout(height=320, paper_bgcolor='white')
fig.show()

print("\n📌 Darker = higher sales. Q4 (Oct-Nov-Dec) consistently the darkest.")



📌 Darker = higher sales. Q4 (Oct-Nov-Dec) consistently the darkest.


### 5.5 Top & Bottom Sub-Categories  
**Question:** Which products are most profitable and which are losing the most?


In [30]:
# ── Horizontal bar: profit by sub-category ────────────────────────────────────
subcat_profit = (
    df.groupby(['category','sub_category'])['profit']
    .sum()
    .reset_index()
    .sort_values('profit')
)

# Color: red for loss, blue for profit
bar_colors = ['#DC2626' if p < 0 else '#2563EB' for p in subcat_profit['profit']]

fig = go.Figure(go.Bar(
    x=subcat_profit['profit'],
    y=subcat_profit['sub_category'],
    orientation='h',
    marker_color=bar_colors,
    text=[f'${p:,.0f}' for p in subcat_profit['profit']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Profit: $%{x:,.0f}<extra></extra>'
))

fig.add_vline(x=0, line_color='gray')

fig.update_layout(
    title='📊 Profit by Sub-Category (Red = Loss)',
    xaxis_title='Total Profit ($)',
    yaxis_title='',
    height=520,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_xaxes(showgrid=True, gridcolor='#F3F4F6')
fig.update_yaxes(showgrid=False)

fig.show()
print("\n📌 Copiers & Phones are star performers. Tables & Bookcases are destroying value.")



📌 Copiers & Phones are star performers. Tables & Bookcases are destroying value.


---
## ✅ Summary & Next Steps

### Key Findings
| # | Insight | Impact |
|---|---------|--------|
| 1 | Discounts >50% = 100% loss rate | 🔴 High |
| 2 | Tables sub-category is unprofitable | 🔴 High |
| 3 | West region leads; Central underperforms | 🟡 Medium |
| 4 | Sales growing YoY but margin declining | 🟡 Medium |
| 5 | Q4 is peak season but prone to over-discounting | 🟡 Medium |

### Recommended Actions
1. **Cap discounts at 40%** — no exception for any category
2. **Review Tables product line** — reprice or discontinue loss-makers  
3. **Audit Central region** — understand why margin lags other regions
4. **Q4 strategy** — build inventory, resist discounting (demand is already high)
5. **Double down on Copiers** — highest margin sub-category


